In [11]:
from glob import glob

# 사용할 PDF 파일 확인하기
for g in glob('../data/*.pdf'):
    print(g)


../data\2040_seoul_plan.pdf
../data\OneNYC_2050_Strategic_Plan.pdf


In [12]:
# read_pdf_and_split_text 함수 생성: PDF 파일을 텍스트로 읽고 청크 단위로 텍스트를 잘라 리스트를 담아 반환
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def read_pdf_and_split_text(pdf_path, chunk_size=1000, chunk_overlap=100):
    """
    주어진 PDF 파일을 읽고 텍스트를 분할합니다.

    매개변수:
        pad_path (str): PDF 파일의 경로
        chunk_size (int, 선택적): 각 텍스트 청크의 크기, 기본값은 1000
        chunk_overlap (int, 선택적): 청크 간 중복 문자 수, 기본값은 100
    반환값:
        list: 분할된 텍스트 청크의 리스트
    """
    print(f"PDF: {pdf_path}------------------------")
    
    pdf_loader = PyPDFLoader(pdf_path)
    data_from_pdf = pdf_loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    splits = text_splitter.split_documents(data_from_pdf)

    print(f"Number of chunks: {len(splits)}\n")
    return splits

In [15]:
# vectorstore 설정하고 청크 처리하기
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import os

##### veectorstore 설정 #####
embedding = OpenAIEmbeddings(model="text-embedding-3-large")

persist_directory='../chroma_store'

if os.path.exists(persist_directory):
    print('Loading existing Chroma store')
    vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embedding)

else:
    print("Creating new Chroma store")

    vectorstore = None
    for g in glob('../data/*.pdf'):
        chunks = read_pdf_and_split_text(g)
        # 100 개씩 나눠서 저장
        for i in range(0, len(chunks), 100):
            if vectorstore is None:
                vectorstore = Chroma.from_documents(
                    documents=chunks[i:i+100],
                    embedding=embedding,
                    persist_directory=persist_directory
                )
            else:
                vectorstore.add_documents(
                    documents=chunks[i:i+100],
                )

Creating new Chroma store
PDF: ../data\2040_seoul_plan.pdf------------------------
Number of chunks: 308

PDF: ../data\OneNYC_2050_Strategic_Plan.pdf------------------------
Number of chunks: 1023



In [16]:
# 관련 문서 검색하기
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

chunks = retriever.invoke("서울 온실가스 저감 계획")

for chunk in chunks:
    print(chunk.metadata)
    print(chunk.page_content)
    print("--------------------------------")

{'creator': 'Hwp 2020 11.0.0.5178', 'creationdate': '2024-12-12T18:16:11+09:00', 'producer': 'Hancom PDF 1.3.0.542', 'author': 'SI', 'page_label': '64', 'source': '../data\\2040_seoul_plan.pdf', 'moddate': '2024-12-12T18:16:11+09:00', 'page': 63, 'pdfversion': '1.4', 'total_pages': 205}
56제2장 미래상과 목표
6. 미래위기를 준비하는, ‘탄소중립 안전도시 구축’1) 배경전(全) 지구적인 기후변화에 대응하기 위한 대도시 차원의 대응 필요Ÿ서울시 2017년 온실가스 배출량은 46,685천 톤CO2eq로 2005년 배출량에 비해 5.6%(276만 톤CO2eq) 감소하였으며, 서울의 1인당 온실가스 배출량은 4.7CO2eq로 국가의 13.8톤CO2eq에 비해 낮은 수준이다. 또한 2014년 기준 서울의 1인당 온실가스 배출량은 4.5CO2eq로 세계 주요 도시에 비해 낮은 수준을 보인다.7)
[그림 2-26] 세계 주요 도시 1인당 온실가스 배출량   자료: World Bank, 2014Ÿ그럼에도 불구하고, 기후변화 문제는 21세기에 전 지구적으로 가장 위중한 영향을 미칠 것으로 예상되는바, 탄소중립과 기후위기 대응은 국제·국가·도시의 핵심 정책으로 다루어져야 한다.Ÿ파리협정(2015) 이후, 우리나라를 비롯한 미국, 유럽연합 등 세계 136개국이 탄소중립을 선언하였으며, 서울시를 비롯한 뉴욕, 런던 등과 같은 많은 글로벌 도시에서 탄소중립을 시정의 핵심과제로 추진 중이다.Ÿ우리나라는 2020년 ‘한국판 그린뉴딜’을 발표하며 공식적으로 2050 탄소중립을 선언하였으며 서울시 역시 2020년 ‘그린뉴딜 추진을 통한 2050 온실가스 감축 전략’을 발표하고 탄소중립 선언에 따른 부문별 사업을 적극 추진하고 있다.-서울시는 2050년 탄소중립을 목

In [17]:
# 챗봇으로 활용하기 위한 언어 모델 설정
from langchain_openai import ChatOpenAI

# 모델 초기화
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
model.invoke("안녕하세요.")

AIMessage(content='안녕하세요! 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 10, 'total_tokens': 20, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_34a54ae93c', 'id': 'chatcmpl-C0iqRka9gcdnWnB1gkYG24m31G8B5', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--8ffbdd9e-8659-4903-8809-33af8b37311c-0', usage_metadata={'input_tokens': 10, 'output_tokens': 10, 'total_tokens': 20, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [18]:
# 라우터 설정하기
from langchain_core.prompts import ChatPromptTemplate
from typing import Literal # 문자열 리터럴 타입을 지원하는 typing 모듈
from pydantic import BaseModel, Field

# Data model
class RouteQuery(BaseModel):
    """사용자 쿼리를 가장 관련성이 높은 데이터 소스로 라우트 합니다."""
    
    datasource: Literal["vectorstore", "casual_talk"] = Field(
        '''
        description="""
            사용자 질문에 따라 casual_talk 또는 vectorestore 로 라우팅 하니다."casual_talk"
            - casual_talk: 일상 대화를 위한 데이터 소스. 사용자가 일상적인 질문을 할 때 사용합니다.
            - vectorstore: 사용자 질문에 답하기 위해 RAG 로 vectorstore 검색이 필요한 경우 사용합니다.
        """
        '''
    )

In [19]:
# 사용자 질문을 받고 판단하기
structured_llm_router = model.with_structured_output(RouteQuery)

router_system = """
당신은 사용자의 질문을 vectorestore 또는 casual_talk 로 라우팅하는 전문가입니다.
- vectorstore 에는 서울, 뉴욕의 발전계획과 관련된 문서가 포함되어 있습니다. 이 주제에 대한 질문에는 vectorstore를 사용하십시오.
- 사용자의 질문이 일상 대화에 관련된 경우 casual_talk 을 사용하십시오.
"""

# 시스템 메시지와 사용자의 질문을 포함하는 프롬프트 템플릿 생성
route_prompt = ChatPromptTemplate.from_messages([
    ("system", router_system),
    ("human", "{question}")
])

# 라우터 프롬프트와 구조화된 출력 모델을 결합한 객체
question_router = route_prompt | structured_llm_router

In [20]:
# 결과 출력하기
print(
    question_router.invoke(
        {
            "question": "서울 온실가스 저감 계획에 대해 알려줘"
        }
    )
)

print(question_router.invoke(
    {
        "question": "잘 지냈어?"
    }
))

datasource='vectorstore'
datasource='casual_talk'


In [21]:
# GradeDocumentes 클래스 만들기 : 출력 방식을 yes 또는 no 로만 제한하여 문서의 관련성 판단
from langchain_core.prompts import PromptTemplate

class GradeDocuments(BaseModel):
    """검색된 문서가 질문과 관련성 있는지 yes 또는 no로 평가합니다"""

    binary_score: Literal["yes", "no"] = Field(
        description="문서가 질문과 관련이 있는지 여부를 'yes' 또는 'no'로 평가합니다."
    )

structured_llm_router = model.with_structured_output(GradeDocuments)